In [1]:
from fenics import *
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()
set_log_level(30)

# Parámetros del modelo (sin control)
D_c = float(os.getenv('D_c'))
D_s = float(os.getenv('D_s'))
D_i = float(os.getenv('D_i'))
rc = float(os.getenv('rc'))
rs = float(os.getenv('rs'))
rd = float(os.getenv('rd'))
alpha = float(os.getenv('alpha'))
delta = float(os.getenv('delta'))
beta = float(os.getenv('beta'))
alle = float(os.getenv('alle'))
gamma = float(os.getenv('gamma'))
eta = float(os.getenv('eta'))
mu = float(os.getenv('mu'))  # se ignora en este caso (sin control)

# Mallado / tiempo
nodes_in_xaxis = int(os.getenv('nodes_in_xaxis'))
nodes_in_yaxis = int(os.getenv('nodes_in_yaxis'))
space_size = float(os.getenv('space_size'))
T = float(os.getenv('T'))
dt = float(os.getenv('dt'))
nb = int(os.getenv('nb'))

# Muestreo denso para guardado (issue 6)
sample_rate = float(os.getenv('sample_rate', 0.02))  # menor => matrices más densas
save_images = os.getenv('SAVE_IMAGES', 'N')

# Iteraciones internas (issue 3)
inner_max_iter = int(os.getenv('inner_max_iter', 3))
inner_tol = float(os.getenv('inner_tol', 1e-3))

print(f"sample_rate={sample_rate}, inner_iter={inner_max_iter}, inner_tol={inner_tol}")

sample_rate=0.02, inner_iter=3, inner_tol=0.001


In [2]:
# Ruta de salida
nueva_ruta = os.path.expanduser('~/Drive/Doctorado Erick Serrato/25-I/strong_allee/mu_0')
os.makedirs(nueva_ruta, exist_ok=True)
os.chdir(nueva_ruta)
print(f"Salida a: {os.getcwd()}")

Salida a: /home/erick/Drive/Doctorado Erick Serrato/25-I/strong_allee/mu_0


In [3]:
def create_space_function(space_size, nx, ny):
    mesh = RectangleMesh(Point(0.0, 0.0), Point(space_size, space_size), nx, ny, "right/left")
    V = FunctionSpace(mesh, 'P', 1)
    return mesh, V

def field_to_numpy_array(fenics_field, space_size, step, field_name, block, sample_rate=0.02):
    sample_points = np.linspace(0, space_size, int(space_size / sample_rate) + 1)
    field_array = np.empty((len(sample_points), len(sample_points)), dtype=float)
    for i, val_x in enumerate(sample_points):
        for j, val_y in enumerate(sample_points):
            try:
                valor = fenics_field(val_x, val_y)
            except Exception:
                valor = 0.0
            field_array[j, i] = valor
    field_array = np.nan_to_num(field_array, nan=0.0, posinf=0.0, neginf=0.0)
    fname = f"matrix_{field_name}_{step}_nb_{block}.txt"
    np.savetxt(fname, field_array, delimiter="\t", fmt="%.8e")
    return fname


In [4]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable, axes_size

def plot_fields(field_c, field_s, field_i, block, t):
    plt.figure(figsize=(16, 5))

    # Cáncer
    plt.subplot(1, 3, 1)
    p1 = plot(field_c)
    p1.set_cmap("magma")
    plt.title(f'c at t = {t:.3f}')
    ax = plt.gca()
    divider = make_axes_locatable(ax)
    width = axes_size.AxesY(ax, aspect=1/20.0)
    pad = axes_size.Fraction(0.5, width)
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.colorbar(p1, cax=cax)

    # Sanas
    plt.subplot(1, 3, 2)
    p2 = plot(field_s)
    p2.set_cmap("gray")
    plt.title(f's at t = {t:.3f}')
    ax = plt.gca()
    divider = make_axes_locatable(ax)
    width = axes_size.AxesY(ax, aspect=1/20.0)
    pad = axes_size.Fraction(0.5, width)
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.colorbar(p2, cax=cax)

    # Inmune
    plt.subplot(1, 3, 3)
    p3 = plot(field_i)
    p3.set_cmap("viridis")
    plt.title(f'i at t = {t:.3f}')
    ax = plt.gca()
    divider = make_axes_locatable(ax)
    width = axes_size.AxesY(ax, aspect=1/20.0)
    pad = axes_size.Fraction(0.5, width)
    cax = divider.append_axes("right", size=width, pad=pad)
    plt.colorbar(p3, cax=cax)

    plt.tight_layout(pad=4)
    if save_images == 'Y':
        plt.savefig(f'fields_block_{block}_step_{t:.3f}.png', dpi=200)
    plt.show()
    plt.close()

In [5]:
def solve_dynamics():
    mesh, V = create_space_function(space_size, nodes_in_xaxis, nodes_in_yaxis)
    c = Function(V)
    s = Function(V)
    i = Function(V)

    phi_c = TestFunction(V)
    phi_s = TestFunction(V)
    phi_i = TestFunction(V)

    # Condiciones iniciales aleatorias suaves
    c_n = interpolate(Expression('0.01 + 0.01*rand()', degree=1), V)
    s_n = interpolate(Expression('0.25 + 0.05*rand()', degree=1), V)
    i_n = interpolate(Expression('0.9 + 0.1*rand()', degree=1), V)

    if mu == 0:
        F_c = ((c - c_n) / dt) * phi_c * dx + D_c * dot(grad(c), grad(phi_c)) * dx + rc * c * (1 - c) * ((c - alle) / (1 - alle)) * phi_c * dx - c*(alpha*s**2 + beta*i**2) * phi_c * dx
        F_s = ((s - s_n) / dt) * phi_s * dx + D_s * dot(grad(s), grad(phi_s)) * dx + rs * s * (1 - s) * phi_s * dx - gamma * c**2 * s * phi_s * dx + delta * i**2 * s * phi_s * dx
        F_i = ((i - i_n) / dt) * phi_i * dx + D_i * dot(grad(i), grad(phi_i)) * dx + rd * i * (1 - i) * phi_i * dx + delta * i * s**2 * phi_i * dx - c**2 * i * eta * phi_i * dx
    else:
        F_c = ((c - c_n) / dt) * phi_c * dx + D_c * dot(grad(c), grad(phi_c)) * dx + rc * c * (1 - c) * ((c - alle) / (1 - alle)) * phi_c * dx - c*(alpha*s**2 + beta*i**2) * phi_c * dx - mu * c*(gamma * s**2 + eta*i**2) * phi_c * dx
        F_s = ((s - s_n) / dt) * phi_s * dx + D_s * dot(grad(s), grad(phi_s)) * dx + rs * s * (1 - s) * phi_s * dx - gamma * c**2 * s * phi_s * dx + delta * i**2 * s * phi_s * dx - ((s*c**2*alpha*mu)/2) * phi_s * dx
        F_i = ((i - i_n) / dt) * phi_i * dx + D_i * dot(grad(i), grad(phi_i)) * dx + rd * i * (1 - i) * phi_i * dx + delta * i * s**2 * phi_i * dx - c**2 * i * eta * phi_i * dx - ((i*c**2*beta*mu)/2) * phi_i * dx

    
    J_c = derivative(F_c, c)
    J_s = derivative(F_s, s)
    J_i = derivative(F_i, i)

    solver_c = NonlinearVariationalSolver(NonlinearVariationalProblem(F_c, c, bcs=[], J=J_c))
    solver_s = NonlinearVariationalSolver(NonlinearVariationalProblem(F_s, s, bcs=[], J=J_s))
    solver_i = NonlinearVariationalSolver(NonlinearVariationalProblem(F_i, i, bcs=[], J=J_i))

    for solver in [solver_c, solver_s, solver_i]:
        prm = solver.parameters['snes_solver']
        prm['method'] = 'vinewtonrsls'
        prm['linear_solver'] = 'mumps'
        prm['maximum_iterations'] = 200
        prm['error_on_nonconvergence'] = False

    return solver_c, solver_s, solver_i, c, s, i, c_n, s_n, i_n, V


In [6]:
# Bucle principal con gráficos
for block in range(9, nb + 1):
    t = 0.0
    solver_c, solver_s, solver_i, c, s, i, c_n, s_n, i_n, V = solve_dynamics()
    step = 0

    while t < T + 1e-12:
        # Iteraciones internas de Picard
        c_prev = c.copy(True)
        s_prev = s.copy(True)
        i_prev = i.copy(True)
        for k in range(inner_max_iter):
            solver_c.solve()
            solver_s.solve()
            solver_i.solve()
            diff = (norm(c.vector() - c_prev.vector(), 'l2') + norm(s.vector() - s_prev.vector(), 'l2') + norm(i.vector() - i_prev.vector(), 'l2'))
            c_prev.assign(c)
            s_prev.assign(s)
            i_prev.assign(i)
            if diff < inner_tol:
                break

        # Avanzar estado previo
        c_n.assign(c)
        s_n.assign(s)
        i_n.assign(i)

        # Graficar (solo bloque 1)
        if block == 1:
            plot_fields(c, s, i, block, t)
        # Guardar campos en alta resolución
        ts = f"{t:.3f}"
        field_to_numpy_array(c, space_size, ts, 'c', block, sample_rate=sample_rate)
        field_to_numpy_array(s, space_size, ts, 's', block, sample_rate=sample_rate)
        field_to_numpy_array(i, space_size, ts, 'i', block, sample_rate=sample_rate)

        t += dt
        step += 1
        if block == 1 and step % 50 == 0:
            print(f"block {block} step {step} t={t:.3f}")